# Build Cropped Dataset (caries / no_caries)
This notebook loads COCO-style JSON, maps `image_id -> file_name`, reads images from `xrays/`, crops by `bbox`, and saves crops into:

- `processed_dataset/train/caries`
- `processed_dataset/train/no_caries`

In [9]:
import os
import json
import cv2
import numpy as np

# STEP 1: paths requested
BASE_PATH = "/Users/chennupatigunadeep/Downloads/Capstone/DENTEX/training_data/quadrant-enumeration-disease"
IMG_FOLDER = BASE_PATH + "/xrays"
JSON_PATH = BASE_PATH + "/train_quadrant_enumeration_disease.json"

# Optional fallback when notebook is opened inside BASE_PATH itself
if not os.path.exists(JSON_PATH):
    BASE_PATH = "."
    IMG_FOLDER = BASE_PATH + "/xrays"
    JSON_PATH = BASE_PATH + "/train_quadrant_enumeration_disease.json"

print('BASE_PATH:', BASE_PATH)
print('IMG_FOLDER exists:', os.path.exists(IMG_FOLDER))
print('JSON_PATH exists:', os.path.exists(JSON_PATH))

BASE_PATH: /Users/chennupatigunadeep/Downloads/Capstone/DENTEX/training_data/quadrant-enumeration-disease
IMG_FOLDER exists: True
JSON_PATH exists: True


In [10]:
# Load JSON
with open(JSON_PATH, 'r') as f:
    data = json.load(f)

images = data.get('images', [])
annotations = data.get('annotations', [])
categories_3 = data.get('categories_3', [])

# Map image_id -> file_name
image_id_to_file = {img['id']: img['file_name'] for img in images}

# category_id_3 -> label name
cat3_map = {c['id']: c['name'] for c in categories_3}

print('images:', len(images))
print('annotations:', len(annotations))
print('categories_3:', cat3_map)

images: 705
annotations: 3529
categories_3: {0: 'Impacted', 1: 'Caries', 2: 'Periapical Lesion', 3: 'Deep Caries'}


In [11]:
# STEP 2: load image using JSON mapping (exact style requested)
example_ann = annotations[0]
example_image_id = example_ann['image_id']
file_name = image_id_to_file[example_image_id]

image_path = os.path.join(IMG_FOLDER, file_name)
img = cv2.imread(image_path)

print('Example file_name:', file_name)
print('Image path:', image_path)
print('Loaded image shape:', None if img is None else img.shape)

Example file_name: train_673.png
Image path: /Users/chennupatigunadeep/Downloads/Capstone/DENTEX/training_data/quadrant-enumeration-disease/xrays/train_673.png
Loaded image shape: (1316, 2744, 3)


In [12]:
# Build processed dataset
OUTPUT_BASE = 'processed_dataset'

caries_dir = os.path.join(OUTPUT_BASE, 'train', 'caries')
no_caries_dir = os.path.join(OUTPUT_BASE, 'train', 'no_caries')
os.makedirs(caries_dir, exist_ok=True)
os.makedirs(no_caries_dir, exist_ok=True)

def ann_to_binary_label(ann):
    c3 = ann.get('category_id_3', -1)
    if c3 in [1, 3]:   # Caries + Deep Caries
        return 'caries'
    else:
        return 'no_caries'

saved_counts = {'caries': 0, 'no_caries': 0}
skipped = 0

for idx, ann in enumerate(annotations):
    img_id = ann.get('image_id')
    file_name = image_id_to_file.get(img_id)
    bbox = ann.get('bbox')

    if file_name is None or bbox is None or len(bbox) != 4:
        skipped += 1
        continue

    image_path = os.path.join(IMG_FOLDER, file_name)
    img = cv2.imread(image_path)
    if img is None:
        skipped += 1
        continue

    x, y, w, h = bbox
    x1 = max(int(np.floor(x)), 0)
    y1 = max(int(np.floor(y)), 0)
    x2 = min(int(np.ceil(x + w)), img.shape[1])
    y2 = min(int(np.ceil(y + h)), img.shape[0])

    if x2 <= x1 or y2 <= y1:
        skipped += 1
        continue

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        skipped += 1
        continue

    crop = cv2.resize(crop, (224, 224))

    label = ann_to_binary_label(ann)
    save_dir = os.path.join(OUTPUT_BASE, 'train', label)
    os.makedirs(save_dir, exist_ok=True)

    out_name = f"{img_id}_{ann.get('id', idx)}.png"
    out_path = os.path.join(save_dir, out_name)
    ok = cv2.imwrite(out_path, crop)

    if ok:
        saved_counts[label] += 1
    else:
        skipped += 1

print('Saved counts:', saved_counts)
print('Skipped:', skipped)
print('Output folders:')
print(' -', caries_dir)
print(' -', no_caries_dir)

# Option B (preferred): use class weights during training to handle imbalance
num_classes = 2
total = saved_counts['caries'] + saved_counts['no_caries']

if saved_counts['caries'] > 0 and saved_counts['no_caries'] > 0:
    class_weights = {
        'caries': total / (num_classes * saved_counts['caries']),
        'no_caries': total / (num_classes * saved_counts['no_caries'])
    }
    print('Class weights:', class_weights)
    print('PyTorch tensor order [no_caries, caries]:', [class_weights['no_caries'], class_weights['caries']])
    print('Keras class_weight dict:', {0: class_weights['no_caries'], 1: class_weights['caries']})
else:
    print('Cannot compute class weights because one class has 0 samples.')

Saved counts: {'caries': 2767, 'no_caries': 762}
Skipped: 0
Output folders:
 - processed_dataset/train/caries
 - processed_dataset/train/no_caries
Class weights: {'caries': 0.6376942537043729, 'no_caries': 2.3156167979002626}
PyTorch tensor order [no_caries, caries]: [2.3156167979002626, 0.6376942537043729]
Keras class_weight dict: {0: 2.3156167979002626, 1: 0.6376942537043729}
